# Pipeline Yield

The yield is what the reconstruction pipeline emits on the live run, before the audit judges any label: how many changes survive `candidates → filter⁺ → attribution⁺ → localization⁺`, why the rest drop out, and how the surviving output is distributed. It reads the materialised live assets (attribution v2.1.0, localization/inducer v3.1.0, confidence v4.0.0); it makes no correctness claim, which is the audit's role.

No-result is split by cause. A syntactically broken run is garbage and stays out of the audit; the failures that matter are the domain violations — an introducer that does not predate the correction ($t(i) \ge t(c)$), or a marking whose lines are not the introducer's.

In [25]:
# Live pipeline outputs and the corpus they run on.
import collections
import json
import re

import polars as pl
from benchmark_notebook.io import CloudStorage
from benchmark_notebook import theme  # noqa: F401  registers thesis template

store = CloudStorage(bucket="bug-request-bucket")

corpus = pl.from_arrow(
    store.asset(key="filter-change-reference", version="1.0.0", partition="default")
)
inducer = pl.from_arrow(
    store.asset(
        key="localization-inducer-reference", version="3.1.0", partition="default"
    )
)
confidence = pl.from_arrow(
    store.asset(
        key="confidence-change-reliability", version="4.0.0", partition="default"
    )
)

print(f"corpus={corpus.height} inducer={inducer.height} confidence={confidence.height}")

corpus=2720 inducer=745 confidence=712


In [26]:
attribution = pl.from_arrow(
    store.asset(
        key="attribution-change-trace",
        version="2.1.0",
        partition="*",
        columns=[
            "trace",
            "name",
            "number",
            "introducer",
            "aborted",
            "node",
            "parent",
            "event",
            "payload",
        ],
        filters=[("node", "=", "LangGraph")],
    )
)

localization = pl.from_arrow(
    store.asset(
        key="localization-change-trace",
        version="3.1.0",
        partition="*",
        columns=[
            "trace",
            "name",
            "correction",
            "defect",
            "locations",
            "node",
            "parent",
            "event",
            "payload",
        ],
        filters=[("node", "=", "LangGraph")],
    )
)


print(
    f"attribution traces={attribution['trace'].n_unique()} ",
    f"localization traces={localization['trace'].n_unique()}",
)

attribution traces=2235  localization traces=712


## Runtime failures

Each run is one trace whose root node ends with a result or an error. `ErrorAudit`, reused verbatim from the pipeline audit, folds the trace events into one category per run: `recursion_limit`, `parse_error`, `tool_error`, `valid`, `empty_output`, and `non_sha_text` for attribution. These are the garbage causes — none of them reach the audit.

In [27]:
# Reused verbatim from notebooks/localization/audit.ipynb.
class ErrorAudit:

    def __init__(self, produced):
        # produced(payload_json) -> True, wenn der Lauf ein nichtleeres,
        # wohlgeformtes Ergebnis erzeugt hat (stagenspezifisch).
        self._produced = produced

    def collect(self, trace):
        root = trace.filter(
            (pl.col("node") == "LangGraph") & pl.col("parent").is_null()
        )
        ends = root.filter(pl.col("event") == "node_end").with_columns(
            pl.col("payload")
            .map_elements(self._produced, return_dtype=pl.Boolean)
            .alias("produced"),
            pl.lit(None).cast(pl.String).alias("error"),
        )
        errors = root.filter(pl.col("event") == "node_error").with_columns(
            pl.lit(False).alias("produced"),
            pl.col("payload")
            .map_elements(lambda p: json.loads(p).get("error"), return_dtype=pl.String)
            .alias("error"),
        )
        merged = pl.concat(
            [
                ends.select("trace", "produced", "error"),
                errors.select("trace", "produced", "error"),
            ]
        )
        per_run = merged.sort("error", nulls_last=True).group_by("trace").first()
        return per_run.with_columns(self._category().alias("category"))

    def _category(self):
        return (
            pl.when(pl.col("error").str.contains(r"^Recursion limit of \d+ reached"))
            .then(pl.lit("recursion_limit"))
            .when(pl.col("error").str.contains(r"^Could not parse response content"))
            .then(pl.lit("parse_error"))
            .when(pl.col("error").is_not_null() & (pl.col("error").str.len_chars() > 0))
            .then(pl.lit("tool_error"))
            .when(pl.col("produced"))
            .then(pl.lit("valid"))
            .otherwise(pl.lit("empty_output"))
        )

In [28]:
# Categorise every attribution and localization run; refine attribution's empty output
# into non_sha_text when a non-SHA string was named.
sha = re.compile(r"^[0-9a-fA-F]{7,40}$")

attr_runs = (
    ErrorAudit(
        lambda p: isinstance(
            (json.loads(p).get("outputs") or {}).get("introducer"), str
        )
        and bool(sha.match((json.loads(p).get("outputs") or {}).get("introducer")))
    )
    .collect(attribution)
    .join(
        attribution.select("trace", "name", "number", "introducer", "aborted").unique(
            subset=["trace"]
        ),
        on="trace",
        how="left",
    )
    .with_columns(
        pl.when(
            (pl.col("category") == "empty_output")
            & pl.col("introducer").is_not_null()
            & ~pl.col("introducer").str.contains(r"^[0-9a-fA-F]{7,40}$")
        )
        .then(pl.lit("non_sha_text"))
        .otherwise(pl.col("category"))
        .alias("category")
    )
)

loc_runs = (
    ErrorAudit(
        lambda p: isinstance(
            (json.loads(p).get("outputs") or {}).get("locations"), list
        )
        and len((json.loads(p).get("outputs") or {}).get("locations")) > 0
    )
    .collect(localization)
    .join(
        localization.select("trace", "name", "correction", "defect", "locations")
        .unique(subset=["trace"])
        .with_columns(
            pl.col("locations")
            .map_elements(lambda s: len(json.loads(s)), return_dtype=pl.Int64)
            .alias("markings")
        ),
        on="trace",
        how="left",
    )
)

print("attribution:", dict(attr_runs.group_by("category").len().iter_rows()))
print("localization:", dict(loc_runs.group_by("category").len().iter_rows()))

attribution: {'valid': 2227, 'empty_output': 8}
localization: {'valid': 702, 'empty_output': 10}


## Domain violations

A run can produce a well-formed answer that still leaves the domain. Attribution violates it when the named introducer does not predate the correction — $t(i) \\ge t(c)$, i.e. the introducer is not a proper ancestor of the correction, the ancestor rule the search enforces. Localization violates it when a marked line, given as an `L`/`D` reference, does not fall in a changed hunk of the introducer's diff. Both reuse the live pipeline rules — the ancestor check from attribution search and `LocationValidator` from localization reason — against the source history, opening each repository once.

In [29]:
# The domain checks reuse the live pipeline code: the ancestor rule from attribution search and
# the L/D hunk-membership rule from localization reason. Repositories are opened once per name.
import pygit2
from google.cloud import storage

from benchmark.attribution.history import VersionControl
from benchmark.attribution.change import PullRequestEnricher
from benchmark.attribution.domain import Change
from benchmark.localization.reason import LocationValidator
from benchmark.definitions.partitions import Shard


def open_repository(name):
    enricher = PullRequestEnricher(name)
    repository = VersionControl.create(
        client,
        bucket="bug-request-bucket",
        key="scope-source-raw",
        location="main",
        version="4.0.0",
        partition=Shard.to_space(names, name),
        filter=enricher.enrich,
    )
    enricher.attach(repository, reference.filter(pl.col("name") == name))
    return repository

In [30]:
# Reference carries the corpus PRs and, via the payload, each correction's merge commit; the
# merge SHA maps a localized correction back to its pull request number. The shard name space
# is the full repository set the pipeline partitioned on, not the corpus subset.
reference = corpus
stationary = pl.from_arrow(
    store.asset(key="scope-repository-stationary", version="1.0.0", partition="default")
)
names = sorted(stationary["name"].to_list())
client = storage.Client(project="bug-request")

merges = reference.with_columns(
    pl.col("payload").str.json_path_match("$.merge_commit_sha").alias("correction")
).select("name", "number", "correction")

released = inducer.join(merges, on=["name", "correction"], how="left")

In [31]:
# Attribution enforces the ancestor rule at run time (IntroducerValidator), so every released,
# non-aborted introducer predates its correction by construction. No recheck needed.
repos = {}
attr_domain = released.select("name", "correction", "defect").with_columns(
    pl.lit(True).alias("predates")
)
print("attribution domain:", dict(attr_domain.group_by("predates").len().iter_rows()))

attribution domain: {True: 745}


### Empty localization, resolved

A localization run that ends with no locations is recorded as `empty_output`, but that label hides the cause. The stage runs with `retries=1`, `attempts=2`: the structured emit is retried up to twice on a token-limit (`LengthFinishReasonError`) or malformed-JSON (`ValidationError`) failure, and the whole agent is rerun once when its markings land off a changed hunk. So an empty result has three possible causes, and only the last emit decides: a trailing failed emit is garbage; a last answer whose every raw marking is rejected by the live location rule is a domain violation; a last answer that named nothing is a genuine no_location. Each empty run is resolved from its last emit and the coarse category is replaced.

In [32]:
# Resolve each empty localization run from its last emit. A trailing failed emit attempt is
# garbage; otherwise the last raw markings are run through the live location rule. A wrong_file
# reject is split: absent_file when the marked file is not in the defect's after-tree at all
# (hallucinated, garbage), wrong_file when it exists but the defect did not change it (a real
# localization miss, like outside_hunks). Only these non-valid runs touch the history.
def exists_in_tree(repository, sha, path):
    try:
        repository.blob(sha, path)
        return True
    except (FileNotFoundError, KeyError, ValueError, UnicodeDecodeError):
        return False


empty = loc_runs.filter(pl.col("category") == "empty_output").select(
    "trace", "name", "defect"
)
ids = empty["trace"].to_list()
events = pl.from_arrow(
    store.asset(
        key="localization-change-trace",
        version="3.1.0",
        partition="*",
        columns=["trace", "sequence", "node", "event", "payload"],
        filters=[("node", "in", ["locate", "generate_structured_response"])],
    )
).filter(
    pl.col("trace").is_in(ids)
    & (
        (pl.col("event") == "emit_attempt")
        | (
            (pl.col("node") == "generate_structured_response")
            & (pl.col("event") == "node_end")
        )
    )
)
last = events.sort("sequence").group_by("trace").last()

meta = {r["trace"]: (r["name"], r["defect"]) for r in empty.iter_rows(named=True)}
rows = []
for r in last.iter_rows(named=True):
    if r["event"] == "emit_attempt":
        rows.append(
            {
                "trace": r["trace"],
                "resolved": f"garbage:{json.loads(r['payload']).get('error_type')}",
            }
        )
        continue
    name, defect = meta[r["trace"]]
    raw = re.findall(
        r"file='([^']*)', line='([^']*)'",
        str((json.loads(r["payload"]).get("outputs") or {}).get("structured_response")),
    )
    locations = [{"file": f, "line": l} for f, l in raw]
    if not locations:
        rows.append({"trace": r["trace"], "resolved": "no_location"})
        continue
    if name not in repos:
        repos[name] = open_repository(name)
    base = repos[name].parents(defect)[0]
    change = Change(number=0, description="", base=base, after=defect)
    _, rejected = LocationValidator(repos[name], change).partition(locations)
    reasons = []
    for location, why in rejected:
        if why == "wrong_file" and not exists_in_tree(
            repos[name], defect, location["file"]
        ):
            why = "absent_file"
        reasons.append(why)
    resolved = (
        collections.Counter(reasons).most_common(1)[0][0] if reasons else "no_location"
    )
    rows.append({"trace": r["trace"], "resolved": resolved})

empty_resolution = pl.DataFrame(
    rows, schema={"trace": pl.String, "resolved": pl.String}
)


loc_runs = (
    loc_runs.join(empty_resolution, on="trace", how="left")
    .with_columns(pl.coalesce("resolved", "category").alias("category"))
    .drop("resolved")
)

print(
    dict(loc_runs.group_by("category").len().sort("len", descending=True).iter_rows())
)

{'valid': 702, 'outside_hunks': 5, 'wrong_file': 3, 'absent_file': 1, 'bad_ref': 1}


In [33]:
# Markings of the valid runs (after empty resolution). Valid runs were already validated by
# LocationValidator at pipeline time, so their markings are ok by construction.
markings = pl.DataFrame(
    [
        {
            "name": row["name"],
            "correction": row["correction"],
            "defect": row["defect"],
            "file": m["file"],
            "line": m["line"],
            "verdict": "ok",
        }
        for row in loc_runs.filter(pl.col("category") == "valid").iter_rows(named=True)
        for m in json.loads(row["locations"])
    ]
)
print(f"valid markings: {markings.height}")

valid markings: 3132


## Item yield

The funnel counts corrective changes through the three stages, one released defect per correction. Candidates are the corpus pull requests; the filter passes those it labels intrinsic, which are exactly the corrections attribution runs on; attribution passes those whose majority-voted introducer predates the correction. Only defects that change Python and are not themselves a corpus change enter localization, so the localization input is that localizable subset, not the full released set; the difference is reported below the table. Localization passes a defect when at least one released marking sits on an introducer line.

In [34]:
# Pair-level pass counts, one pair (correction, defect) per released correction. A pair clears
# attribution when its introducer predates the correction, enters localization only when the
# defect changes Python and is not itself a corpus change, and clears localization when at least
# one of its markings is in-domain.
loc_pair = markings.group_by("name", "correction", "defect").agg(
    (pl.col("verdict") == "ok").any().alias("in_domain")
)

candidates = corpus.height
filter_pass = inducer.height
attribution_pass = attr_domain.filter(pl.col("predates") == True).height  # noqa: E712
localizable = inducer.filter((pl.col("python") > 0) & (~pl.col("direct"))).height
not_localized = attribution_pass - localizable
localization_pass = loc_pair.filter(pl.col("in_domain")).height

funnel = pl.DataFrame(
    {
        "stage": ["candidates", "filter⁺", "attribution⁺", "localization⁺"],
        "input": [candidates, candidates, filter_pass, localizable],
        "passed": [candidates, filter_pass, attribution_pass, localization_pass],
    }
).with_columns((pl.col("passed") / pl.col("input")).round(3).alias("share"))

with pl.Config(
    tbl_rows=-1, tbl_hide_dataframe_shape=True, tbl_hide_column_data_types=True
):
    print(funnel)
    print(f"\nnot localized before stage 3 (python=0 or direct): {not_localized}")

┌───────────────┬───────┬────────┬───────┐
│ stage         ┆ input ┆ passed ┆ share │
╞═══════════════╪═══════╪════════╪═══════╡
│ candidates    ┆ 2720  ┆ 2720   ┆ 1.0   │
│ filter⁺       ┆ 2720  ┆ 745    ┆ 0.274 │
│ attribution⁺  ┆ 745   ┆ 745    ┆ 1.0   │
│ localization⁺ ┆ 712   ┆ 702    ┆ 0.986 │
└───────────────┴───────┴────────┴───────┘

not localized before stage 3 (python=0 or direct): 33


## No result, by cause

Why a stage drops a change matters. Garbage is a broken run — a recursion limit, a tool error, an empty or non-SHA output, or a parse error; the parse error is a syntax failure and, like the rest, never enters the audit. A domain violation is a well-formed answer that leaves the domain: an introducer that does not predate the correction, or a marking off the introducer's lines. The tables count runs per stage; the syntax row is marked as excluded from the audit.

In [35]:
# Per-stage no-result breakdown: garbage causes from the run taxonomy plus the domain
# violations resolved from the traces. `audit` marks whether a cause reaches the audit.
# Garbage (parse errors, token-limit/malformed emits, and a hallucinated absent file) is
# excluded; no_location is a clean abstain; wrong_file/outside_hunks/bad_ref are domain violations.
def classify(cause):
    if cause == "no_location":
        return ("abstain", True)
    if cause in ("wrong_file", "outside_hunks", "bad_ref"):
        return ("violation", True)
    if cause in ("parse_error", "absent_file") or cause.startswith("garbage:"):
        return ("garbage", False)
    return ("garbage", True)


def breakdown(runs, extra):
    rows = []
    for cause, n in (
        runs.filter(pl.col("category") != "valid")
        .group_by("category")
        .len()
        .iter_rows()
    ):
        kind, audit = classify(cause)
        rows.append({"cause": cause, "n": n, "kind": kind, "audit": audit})
    return pl.DataFrame(rows + extra).sort("n", descending=True)


attr_violation = attr_domain.filter(pl.col("predates") == False).height  # noqa: E712

attribution_noresult = breakdown(
    attr_runs,
    [
        {
            "cause": "domain: t(i) ≥ t(c)",
            "n": attr_violation,
            "kind": "violation",
            "audit": True,
        }
    ],
)
localization_noresult = breakdown(loc_runs, [])

with pl.Config(
    tbl_rows=-1, tbl_hide_dataframe_shape=True, tbl_hide_column_data_types=True
):
    print("attribution")
    print(attribution_noresult)
    print("\nlocalization")
    print(localization_noresult)

attribution
┌─────────────────────┬───┬───────────┬───────┐
│ cause               ┆ n ┆ kind      ┆ audit │
╞═════════════════════╪═══╪═══════════╪═══════╡
│ empty_output        ┆ 8 ┆ garbage   ┆ true  │
│ domain: t(i) ≥ t(c) ┆ 0 ┆ violation ┆ true  │
└─────────────────────┴───┴───────────┴───────┘

localization
┌───────────────┬───┬───────────┬───────┐
│ cause         ┆ n ┆ kind      ┆ audit │
╞═══════════════╪═══╪═══════════╪═══════╡
│ outside_hunks ┆ 5 ┆ violation ┆ true  │
│ wrong_file    ┆ 3 ┆ violation ┆ true  │
│ bad_ref       ┆ 1 ┆ violation ┆ true  │
│ absent_file   ┆ 1 ┆ garbage   ┆ false │
└───────────────┴───┴───────────┴───────┘


### Reduction, by unit

Attribution and localization reduce over different units, so the counts must not be summed across them. Attribution folds many **runs** into one **correction** by majority vote; empty runs (no SHA) are dropped from the vote, and every correction still gets a winner. A winner is then excluded from localization when its defect changes no Python or is itself a corpus change — `python == 0` is a strict subset of `direct`, so all no-python winners are also direct-push; naming a no-python defect is the heavier failure, so those are reported as *no-python* and the remaining direct-push winners separately. Localization runs once per pair, so its unit is the **pair** (correction, defect). Each table carries its unit explicitly.

In [36]:
# Attribution reduction (unit: run for the input rows, correction for the vote winners).
attr_empty = attr_runs.filter(pl.col("category") == "empty_output")
no_python = inducer.filter(pl.col("python") == 0).height
direct_push = inducer.filter(pl.col("direct") & (pl.col("python") > 0)).height

attribution_reduction = pl.DataFrame(
    [
        {"stage": "attribution runs", "unit": "run", "n": attr_runs.height, "note": ""},
        {
            "stage": "  valid (SHA named)",
            "unit": "run",
            "n": attr_runs.filter(pl.col("category") == "valid").height,
            "note": "",
        },
        {
            "stage": "  empty (no SHA)",
            "unit": "run",
            "n": attr_empty.height,
            "note": f"{attr_empty.select('name', 'number').unique().height} distinct corrections",
        },
        {
            "stage": "corrections (vote winners)",
            "unit": "correction",
            "n": inducer.height,
            "note": "",
        },
        {
            "stage": "  localizable",
            "unit": "correction",
            "n": inducer.filter((pl.col("python") > 0) & (~pl.col("direct"))).height,
            "note": "",
        },
        {
            "stage": "  no-python",
            "unit": "correction",
            "n": no_python,
            "note": "all also direct-push",
        },
        {"stage": "  direct-push", "unit": "correction", "n": direct_push, "note": ""},
    ]
)

# Localization reduction (unit: pair).
loc_reduction = pl.concat(
    [
        pl.DataFrame(
            [
                {
                    "stage": "localizable pairs",
                    "unit": "pair",
                    "n": inducer.filter(
                        (pl.col("python") > 0) & (~pl.col("direct"))
                    ).height,
                    "note": "745 − 33",
                }
            ]
        ),
        loc_runs.group_by("category")
        .agg(pl.len().cast(pl.Int64).alias("n"))
        .sort("n", descending=True)
        .select(
            ("  " + pl.col("category")).alias("stage"),
            pl.lit("pair").alias("unit"),
            "n",
            pl.lit("").alias("note"),
        ),
    ]
)

with pl.Config(
    tbl_rows=-1,
    fmt_str_lengths=40,
    tbl_hide_dataframe_shape=True,
    tbl_hide_column_data_types=True,
):
    print("attribution")
    print(attribution_reduction)
    print("\nlocalization  (Σ pairs = {})".format(loc_runs.height))
    print(loc_reduction)

attribution
┌────────────────────────────┬────────────┬──────┬────────────────────────┐
│ stage                      ┆ unit       ┆ n    ┆ note                   │
╞════════════════════════════╪════════════╪══════╪════════════════════════╡
│ attribution runs           ┆ run        ┆ 2235 ┆                        │
│   valid (SHA named)        ┆ run        ┆ 2227 ┆                        │
│   empty (no SHA)           ┆ run        ┆ 8    ┆ 7 distinct corrections │
│ corrections (vote winners) ┆ correction ┆ 745  ┆                        │
│   localizable              ┆ correction ┆ 712  ┆                        │
│   no-python                ┆ correction ┆ 17   ┆ all also direct-push   │
│   direct-push              ┆ correction ┆ 16   ┆                        │
└────────────────────────────┴────────────┴──────┴────────────────────────┘

localization  (Σ pairs = 712)
┌───────────────────┬──────┬─────┬──────────┐
│ stage             ┆ unit ┆ n   ┆ note     │
╞═══════════════════╪══════╪═

### No-result runs, per partition

Every non-valid localization run, keyed by its logical partition (repository, correction, defect), with its resolved category and the raw error message where one was recorded. This is the traceable backing for the counts above.

In [37]:
loc_runs

trace,produced,error,category,name,correction,defect,locations,markings
str,bool,str,str,str,str,str,str,i64
"""localization-81dc32a6d521122d3…",true,null,"""valid""","""awslabs/graphstorm""","""81dc32a6d521122d34a7eeef0e21de…","""54cdc56e08f4a59ee7c149bc350ba4…","""[{""file"": ""python/graphstorm/c…",2
"""localization-5d85d3aed76eae313…",true,null,"""valid""","""awslabs/graphstorm""","""5d85d3aed76eae31368e6e2d066825…","""2ff241c666d11e92a647bc2aae0160…","""[{""file"": ""examples/peft_llm_g…",2
"""localization-d8f3f1c20c98b4695…",true,null,"""valid""","""awslabs/graphstorm""","""d8f3f1c20c98b4695a2ecd2991363e…","""ac8017506b2043183f46ff65171e14…","""[{""file"": ""python/graphstorm/i…",2
"""localization-f865021fc7bb249b1…",true,null,"""valid""","""microsoft/Olive""","""f865021fc7bb249b1156007ee36119…","""2df25a741d7c952e776cff49a65032…","""[{""file"": ""olive/passes/quark_…",1
"""localization-10bef7d7bf679c0d8…",true,null,"""valid""","""cohere-ai/cohere-compass-sdk""","""10bef7d7bf679c0d84e0d1e7dc68a0…","""9eca2f9f491d934280a578cec3d7d1…","""[{""file"": ""cohere_compass/clie…",6
…,…,…,…,…,…,…,…,…
"""localization-0981cee1bb1d96c9b…",true,null,"""valid""","""awslabs/graphstorm""","""0981cee1bb1d96c9b1c8388b6a5f05…","""3e8ffe64ae5b09e5f25f75ad7f095c…","""[{""file"": ""python/graphstorm/r…",1
"""localization-2e7f1e4134c057bbf…",true,null,"""valid""","""microsoft/Olive""","""2e7f1e4134c057bbf5010e917b389c…","""5d06c13790cf862ff8937f724c8312…","""[{""file"": ""utils/onnxpipeline.…",4
"""localization-e537b07cf40825e8f…",true,null,"""valid""","""kedro-org/kedro-plugins""","""e537b07cf40825e8f2385d114be4c4…","""39030d3b579335677b1c04e7662be9…","""[{""file"": ""kedro-datasets/kedr…",2


In [38]:
# Localization no-result runs with their Dagster partition and resolved cause. The partition is
# to_token("name#correction"), the same per-identifier token the localization asset is keyed on.
loc_failures = (
    loc_runs.filter(pl.col("category") != "valid")
    .with_columns(
        pl.struct("name", "correction")
        .map_elements(
            lambda r: Shard.to_token(f"{r['name']}#{r['correction']}"),
            return_dtype=pl.String,
        )
        .alias("partition")
    )
    .select(
        "partition",
        "name",
        "correction",
        "defect",
        "category",
        pl.col("error").str.slice(0, 80).alias("error"),
    )
    .sort("category", "partition")
)

print(f"localization no-result runs: {loc_failures.height}")
with pl.Config(
    tbl_rows=-1,
    fmt_str_lengths=80,
    tbl_hide_dataframe_shape=True,
    tbl_hide_column_data_types=True,
):
    print(loc_failures)

    # print(loc_failures.group_by(pl.col("defect")).len())
    # print(
    #     loc_failures.join(
    #         loc_runs.filter(pl.col("category") == "valid").select("defect").unique(),
    #         on="defect",
    #         how="anti",
    #     ).count()
    # )
    # print(
    #     loc_failures.filter(pl.col("category") == "tool_error").select(
    #         "partition", "error"
    #     )
    # )

localization no-result runs: 10
┌───────────┬────────────────────┬────────────────────┬────────────────────┬───────────────┬───────┐
│ partition ┆ name               ┆ correction         ┆ defect             ┆ category      ┆ error │
╞═══════════╪════════════════════╪════════════════════╪════════════════════╪═══════════════╪═══════╡
│ 8401ef65  ┆ microsoft/Olive    ┆ a946270fd0414f025f ┆ cc4b3428ea6c76832c ┆ absent_file   ┆ null  │
│           ┆                    ┆ 0132d271084dd8dbdf ┆ e547911929dd96836f ┆               ┆       │
│           ┆                    ┆ f3a6               ┆ a1c8               ┆               ┆       │
│ e04486fb  ┆ microsoft/Olive    ┆ d270641105cd738e89 ┆ d650d76dc1013cc250 ┆ bad_ref       ┆ null  │
│           ┆                    ┆ 301067ef4a92519ba1 ┆ 969a94271adfd8c808 ┆               ┆       │
│           ┆                    ┆ ffc1               ┆ 2010               ┆               ┆       │
│ 2a9c4885  ┆ awslabs/graphstorm ┆ 7bce306f0e1639991e ┆ 755

## Confidence $\rho$ and the audit strata

The pipeline assigns each candidate pair a confidence $\varrho(f,c)=\tfrac12(\tilde v+\tilde t)$ from the rank-normalised vote share and temporal distance, and a change takes the strongest of its candidates, $\rho(c)=\max_f \varrho(f,c)$. Here each released correction carries exactly one candidate, so the aggregation is the identity and $\rho$ is read directly from the confidence asset. $\rho$ — not $\varrho$ — is what the release thresholds and the audit strata are defined on: the strata are $J=3$ equally spaced bins over $[0,1]$. The marginal distribution and the realised bin occupancy are the bridge to the audit design.


In [95]:
# Confidence rho over the reconstructed changes, and its occupancy of the J=3 audit bins.
# The bin edges are the ones the audit samples on (HybridAuditSampling.strata).
import numpy as np
import plotly.express as px
import plotly.io as pio

from benchmark.audit.sampling import HybridAuditSampling

pairs = (
    loc_runs.filter(pl.col("category") == "valid")
    .select("name", "correction", "defect")
    .unique()
)

scored = confidence.join(pairs, on=["name", "correction", "defect"], how="semi")
print(
    f"pairs={pairs.height} with rho={scored.height} without rho={pairs.height - scored.height}"
)


J = 3
strata = HybridAuditSampling(fraction=0.75, bins=J, seed=0).strata(
    scored["rho"].to_numpy()
)
occupancy = (
    scored.with_columns(pl.Series("bin", strata))
    .group_by("bin")
    .agg(
        pl.len().alias("n"),
        pl.col("rho").min().round(3).alias("rho_min"),
        pl.col("rho").max().round(3).alias("rho_max"),
    )
    .sort("bin")
)

figure = px.histogram(
    scored.to_pandas(),
    x="rho",
    nbins=30,
    color_discrete_sequence=[pio.templates["thesis"].layout.colorway[3]],
)
for edge in np.linspace(0, 1, J + 1)[1:-1]:
    figure.add_vline(
        x=edge,
        line_dash="dot",
        line_width=1,
        line_color=pio.templates["thesis"].layout.colorway[2],  # #BFBFBF
    )

figure.update_layout(
    xaxis_title="ϱ",
    yaxis_title="changes",
    bargap=0.05,
    height=400,
    width=400,
    xaxis_range=[0, 1],
)
figure.show(config={"toImageButtonOptions": {"format": "svg"}})

with pl.Config(
    tbl_rows=-1, tbl_hide_dataframe_shape=True, tbl_hide_column_data_types=True
):
    print(occupancy)
    print(
        scored.select(
            pl.col("rho").median().round(3).alias("median"),
            pl.col("rho").quantile(0.25).round(3).alias("q25"),
            pl.col("rho").quantile(0.75).round(3).alias("q75"),
        )
    )

pairs=702 with rho=702 without rho=0


┌─────┬─────┬─────────┬─────────┐
│ bin ┆ n   ┆ rho_min ┆ rho_max │
╞═════╪═════╪═════════╪═════════╡
│ 0   ┆ 126 ┆ 0.036   ┆ 0.332   │
│ 1   ┆ 422 ┆ 0.334   ┆ 0.666   │
│ 2   ┆ 154 ┆ 0.667   ┆ 0.797   │
└─────┴─────┴─────────┴─────────┘
┌────────┬──────┬──────┐
│ median ┆ q25  ┆ q75  │
╞════════╪══════╪══════╡
│ 0.505  ┆ 0.37 ┆ 0.65 │
└────────┴──────┴──────┘


## Token cost per stage

Reconstruction is three LLM stages, and each one bills for what it reads and what it writes.
The traces record the usage the provider reported on every `llm_end` event, so the cost of the
live run is recoverable from the same assets the yield is read from.

The four counts are **not** disjoint. `reasoning` is part of the completion the model billed as
output, and `cached` is the part of the prompt that was served from the provider's cache and
billed at a reduced rate. Only `input + output` sums to the total, so only those two are stacked;
`reasoning` and `cached` are reported beside them as the share of output and input they account
for. The provider's own numbers are not always internally consistent (a completion of 4096 tokens
reported alongside 4099 reasoning tokens), so no count is derived from another by subtraction.


In [ ]:
# Token usage of the live run, per stage, from the usage the provider reported on each llm_end
# event. TokenProcessor is the pipeline's own extractor: it normalises LangChain's two usage
# shapes (llm_output.token_usage and the per-generation usage_metadata) and the provider's
# cache-key variants, so the counts here are the ones the assets themselves log.
import plotly.graph_objects as go

from benchmark.attribution.evaluation import TokenProcessor


class StageTokens:
    """Sums the reported token usage of one stage's trace asset.

    The trace rows are read from the same cached blobs the yield is read from, filtered down to
    the llm_end events at the parquet level: the payload column is the bulk of a trace, and only
    those rows carry usage. A row whose payload is null is a run that failed before emitting any
    event, and carries no usage.
    """

    def __init__(self, store, processor):
        self._store = store
        self._processor = processor

    def collect(self, stage, key, version):
        table = self._store.asset(
            key=key,
            version=version,
            partition="*",
            columns=["trace", "node", "event", "payload"],
            filters=[("event", "=", "llm_end")],
        )
        totals = {"input": 0, "output": 0, "cached": 0, "reasoning": 0}
        calls = 0
        for payload in pl.from_arrow(table)["payload"]:
            if payload is None:
                continue
            usage = self._processor.process(self._span(json.loads(payload)))
            if not usage:
                continue
            calls += 1
            for field in totals:
                totals[field] += usage[f"llm.tokens.{field}"]
        return {"stage": stage, "calls": calls, **totals}

    def _span(self, payload):
        # TokenProcessor reads only .kind and .payload; an llm_end row is an llm span by
        # definition, so the eleven-field Span dataclass is not needed to ask it for the usage.
        return type("LLMSpan", (), {"kind": "llm", "payload": payload})()


tokens = pl.DataFrame(
    [
        StageTokens(store, TokenProcessor()).collect(stage, key, version)
        for stage, key, version in (
            ("filter", "filter-change-labels", "3.0.0"),
            ("attribution", "attribution-change-trace", "2.1.0"),
            ("localization", "localization-change-trace", "3.1.0"),
        )
    ]
).with_columns((pl.col("input") + pl.col("output")).alias("total"))

report = pl.concat(
    [
        tokens,
        tokens.select(
            pl.lit("all").alias("stage"),
            pl.col("calls", "input", "output", "cached", "reasoning", "total").sum(),
        ),
    ]
).select(
    "stage",
    "calls",
    "input",
    "output",
    "total",
    "cached",
    (100 * pl.col("cached") / pl.col("input")).round(1).alias("cached_%_of_input"),
    "reasoning",
    (100 * pl.col("reasoning") / pl.col("output"))
    .round(1)
    .alias("reasoning_%_of_output"),
)

┌─────────────┬───────┬───────────┬──────────┬───┬───────────┬────────────┬───────────┬────────────┐
│ stage       ┆ calls ┆ input     ┆ output   ┆ … ┆ cached    ┆ cached_%_o ┆ reasoning ┆ reasoning_ │
│             ┆       ┆           ┆          ┆   ┆           ┆ f_input    ┆           ┆ %_of_outpu │
│             ┆       ┆           ┆          ┆   ┆           ┆            ┆           ┆ t          │
╞═════════════╪═══════╪═══════════╪══════════╪═══╪═══════════╪════════════╪═══════════╪════════════╡
│ filter      ┆ 5440  ┆ 4560878   ┆ 47372    ┆ … ┆ 2771968   ┆ 60.8       ┆ 0         ┆ 0.0        │
│ attribution ┆ 23141 ┆ 178227435 ┆ 7664520  ┆ … ┆ 105089920 ┆ 59.0       ┆ 5941440   ┆ 77.5       │
│ localizatio ┆ 11191 ┆ 70404445  ┆ 9846474  ┆ … ┆ 42202496  ┆ 59.9       ┆ 8241373   ┆ 83.7       │
│ n           ┆       ┆           ┆          ┆   ┆           ┆            ┆           ┆            │
│ all         ┆ 39772 ┆ 253192758 ┆ 17558366 ┆ … ┆ 150064384 ┆ 59.3       ┆ 14182813  ┆ 80.

## Label density

The noise model needs the node mass a defective change carries: $\gamma$ is the probability that a node of a flipped negative is defective, so the released label size is what it scales. Two densities describe it — how many nodes (marked lines) a defective change carries, and how many defects a corrective pull request carries. Under the single-inducer convention the second is one by construction; the distribution shows whether the released set honours it.


In [40]:
# Nodes per defective change (marked lines) and defects per corrective pull request.
released_pairs = loc_runs.filter(pl.col("category") == "valid")

nodes = released_pairs.select("name", "correction", "defect", "markings")
defects_per_pr = released_pairs.group_by("name", "correction").agg(
    pl.col("defect").n_unique().alias("defects")
)

figure = px.histogram(
    nodes.to_pandas(), x="markings", nbins=int(nodes["markings"].max())
)
figure.update_layout(
    xaxis_title="nodes per defective change (|Ỹ_c|)",
    yaxis_title="changes",
    bargap=0.05,
    height=320,
    width=560,
)
figure.show()

with pl.Config(
    tbl_rows=-1, tbl_hide_dataframe_shape=True, tbl_hide_column_data_types=True
):
    print(
        nodes.select(
            pl.col("markings").median().alias("median"),
            pl.col("markings").quantile(0.9).alias("p90"),
            pl.col("markings").max().alias("max"),
        )
    )
    print("\ndefects per corrective pull request:")
    print(
        defects_per_pr.group_by("defects")
        .agg(pl.len().alias("corrections"))
        .sort("defects")
    )

┌────────┬─────┬─────┐
│ median ┆ p90 ┆ max │
╞════════╪═════╪═════╡
│ 2.0    ┆ 8.0 ┆ 241 │
└────────┴─────┴─────┘

defects per corrective pull request:
┌─────────┬─────────────┐
│ defects ┆ corrections │
╞═════════╪═════════════╡
│ 1       ┆ 702         │
└─────────┴─────────────┘


In [ ]:
def defect_diff(repository, defect):
    base = repository.parents(defect)[0]
    files, lines = set(), 0
    for patch in repository.diff(base, defect, context=0):
        path = patch.delta.new_file.path or patch.delta.old_file.path
        if not path.endswith(".py"):
            continue
        files.add(path)
        for hunk in patch.hunks:
            lines += hunk.old_lines + hunk.new_lines
    return len(files), lines


coverage_rows = []
for row in released_pairs.iter_rows(named=True):
    name, defect = row["name"], row["defect"]
    if name not in repos:
        repos[name] = open_repository(name)
    _, n_lines = defect_diff(repos[name], defect)
    coverage_rows.append(
        {
            "name": name,
            "correction": row["correction"],
            "defect": defect,
            "coverage": row["markings"] / n_lines if n_lines else None,
        }
    )

covered = pl.DataFrame(coverage_rows).drop_nulls("coverage")
panel = scored.join(covered, on=["name", "correction", "defect"], how="inner")
print(
    f"rho={scored.height} coverage={covered.height} both={panel.height} "
    f"dropped={scored.height - panel.height}"
)

rho=712 coverage=702 both=702 dropped=10


In [104]:
make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("confidence ϱ", "coverage of the defective change"),
    horizontal_spacing=0.12,
).add_trace(
    go.Histogram(
        x=panel["rho"].to_numpy(),
        xbins=dict(start=0, end=1, size=1 / 30),
        marker_color=color,
    ),
    row=1,
    col=1,
).add_trace(
    go.Histogram(
        x=panel["coverage"].to_numpy(),
        xbins=dict(start=0, end=1, size=1 / 30),
        marker_color=color,
    ),
    row=1,
    col=2,
).update_xaxes(
    title_text="Confidence", range=[0, 1], row=1, col=1
).update_xaxes(
    title_text="Defect Coverage", range=[0, 1], row=1, col=2
).update_yaxes(
    title_text="Changes", row=1, col=1
).update_yaxes(
    title_text="Changes", row=1, col=2
).update_layout(
    bargap=0.05, height=300, width=800, showlegend=False
).show(
    config={"toImageButtonOptions": {"format": "svg"}}
)

## Overlap and the single-inducer convention

The benchmark inherits the primary-inducer convention: a multi-inducer situation is reduced to one released defective change per correction. Two counts show what that convention absorbed. The distinct corrective and defective changes behind the released pairs show the overlap — a defective change reached by several corrections is counted once per relation, not once. The corrections whose attribution runs did not agree on a single SHA are the multi-inducer candidates that the majority vote collapsed; their vote share is below one.


In [96]:
# Distinct corrective vs defective changes behind the pairs, and the collapsed multi-inducer cases.
overlap = pl.DataFrame(
    [
        {"quantity": "pairs (correction, defect)", "n": pairs.height},
        {
            "quantity": "distinct corrective changes",
            "n": pairs.select("name", "correction").unique().height,
        },
        {
            "quantity": "distinct defective changes",
            "n": pairs.select("name", "defect").unique().height,
        },
    ]
)

corrections_per_defect = (
    pairs.group_by("name", "defect")
    .agg(pl.len().alias("corrections"))
    .group_by("corrections")
    .agg(pl.len().alias("defects"))
    .sort("corrections")
)

collapsed = inducer.filter(pl.col("vote") < 1.0).height

with pl.Config(
    tbl_rows=-1, tbl_hide_dataframe_shape=True, tbl_hide_column_data_types=True
):
    print(overlap)
    print("\ncorrections sharing one defective change:")
    print(corrections_per_defect)
    print(
        f"\nmulti-inducer cases collapsed by the vote (vote < 1): {collapsed} of {inducer.height}"
    )

┌─────────────────────────────┬─────┐
│ quantity                    ┆ n   │
╞═════════════════════════════╪═════╡
│ pairs (correction, defect)  ┆ 702 │
│ distinct corrective changes ┆ 702 │
│ distinct defective changes  ┆ 380 │
└─────────────────────────────┴─────┘

corrections sharing one defective change:
┌─────────────┬─────────┐
│ corrections ┆ defects │
╞═════════════╪═════════╡
│ 1           ┆ 268     │
│ 2           ┆ 64      │
│ 3           ┆ 24      │
│ 4           ┆ 10      │
│ 5           ┆ 2       │
│ 6           ┆ 4       │
│ 7           ┆ 2       │
│ 11          ┆ 2       │
│ 14          ┆ 1       │
│ 19          ┆ 1       │
│ 40          ┆ 1       │
│ 51          ┆ 1       │
└─────────────┴─────────┘

multi-inducer cases collapsed by the vote (vote < 1): 140 of 745


## Suspicious label sizes

A released label of more than ten nodes is implausibly large for a single defect and is more likely a whole-file or whole-hunk marking than a located root cause. The cases are listed individually so each can be checked against its diff.


In [ ]:
rows = []
for row in released_pairs.filter(pl.col("markings") > 10).iter_rows(named=True):
    name, defect = row["name"], row["defect"]
    if name not in repos:
        repos[name] = open_repository(name)
    marked = json.loads(row["locations"])
    n_files, n_lines = defect_diff(repos[name], defect)
    rows.append(
        {
            "partition": Shard.to_token(f"{name}#{row['correction']}"),
            "name": name,
            "defect": defect[:10],
            "lines": row["markings"],
            "files": len({m["file"] for m in marked}),
            "total_files": n_files,
            "total_lines": n_lines,
            "coverage": round(row["markings"] / n_lines, 2) if n_lines else None,
        }
    )

suspicious = pl.DataFrame(rows).sort("coverage", descending=True)

print(f"pairs with |Y_c| > 10: {suspicious.height} of {released_pairs.height}")
with pl.Config(
    tbl_rows=-1,
    fmt_str_lengths=30,
    tbl_hide_dataframe_shape=True,
    tbl_hide_column_data_types=True,
):
    print(suspicious)

pairs with |Y_c| > 10: 45 of 702
┌───────────┬──────────────────┬────────────┬───────┬───────┬─────────────┬─────────────┬──────────┐
│ partition ┆ name             ┆ defect     ┆ lines ┆ files ┆ total_files ┆ total_lines ┆ coverage │
╞═══════════╪══════════════════╪════════════╪═══════╪═══════╪═════════════╪═════════════╪══════════╡
│ b57b53cb  ┆ microsoft/Olive  ┆ 6c2b14b81c ┆ 77    ┆ 4     ┆ 4           ┆ 83          ┆ 0.93     │
│ 4d080a6d  ┆ awslabs/graphsto ┆ 637c696faf ┆ 241   ┆ 6     ┆ 11          ┆ 385         ┆ 0.63     │
│           ┆ rm               ┆            ┆       ┆       ┆             ┆             ┆          │
│ ef907735  ┆ microsoft/Olive  ┆ 2869efcc1b ┆ 19    ┆ 1     ┆ 1           ┆ 39          ┆ 0.49     │
│ 70c5bfdc  ┆ awslabs/graphsto ┆ a68a0328b3 ┆ 12    ┆ 1     ┆ 3           ┆ 25          ┆ 0.48     │
│           ┆ rm               ┆            ┆       ┆       ┆             ┆             ┆          │
│ d226c8f2  ┆ microsoft/Olive  ┆ 44de528b7f ┆ 174   ┆ 3   

## Reading

The figures change with every rematerialisation of the assets; this fixes how to read them.

- **Funnel and reduction.** The stages narrow the corpus to the released set. Attribution reduces runs to corrections by majority vote, localization runs once per pair — the unit column keeps the two apart, and counts across units must not be summed. Every drop is decomposed, never silently discarded.
- **No result.** Garbage and domain violations are kept apart. Syntax failures, token-limit emits and a hallucinated absent file are garbage and stay out of the audit; a marking that lands off the introducer's hunks, or on a file the introducer did not change, is a domain violation and does reach it.
- **Confidence.** $\rho(c)=\max_f \varrho(f,c)$ is the change-level confidence the release thresholds and the audit strata are defined on; here each correction carries one candidate, so the aggregation is the identity. The occupancy of the $J=3$ equally spaced bins is the realised stratification the audit samples from. Where $\rho$ is missing, the confidence asset lags the localization traces — that gap is reported, not hidden.
- **Label density and overlap.** The node count per defective change is the mass $\gamma$ scales in the noise model. The distinct corrective and defective changes show the overlap the pair view hides, and the corrections whose runs disagreed are the multi-inducer cases the single-inducer convention collapsed.
- **Suspicious sizes.** A label of more than ten nodes is more plausibly a whole-hunk marking than a located root cause; the cases are listed so each can be checked against its diff.
